

# Complete PDF-to-Database Pipeline with OCR Integration

This notebook demonstrates the complete pipeline from PDF files to extracted data in the database:

## Full Pipeline Overview

0. **arXiv Download**: Download papers directly from arXiv using paper IDs
1. **OCR Processing**: Convert PDF files to markdown using Azure OCR API
2. **Token Analysis**: Check if chunking is needed based on token count
3. **Chunked Analysis**: Process large documents with intelligent chunking
4. **LLM Analysis**: Extract structured data using DeepSeek or Gemini
5. **Database Storage**: Save papers, models, and extraction results to PostgreSQL
6. **Consensus Analysis**: Compare results across multiple runs

## Features:
- **arXiv Integration**: Download papers directly from arXiv using paper IDs
- **OCR Integration**: Process PDF files directly with Azure Mistral OCR
- **Smart Chunking**: Splits large documents using intelligent tokenization
- **Multi-Provider Support**: DeepSeek, Gemini analyzers
- **Database Integration**: Complete storage and retrieval system
- **Progressive Analysis**: Context-aware chunking for better results

## Setup and Import Required Libraries

In [ ]:
import os
import sys
import json
import time
import base64
import requests
from pathlib import Path
from typing import Optional, List

# Add the current directory to sys.path to fix import issues
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# Import arXiv downloader
from arxiv_tools.arxiv_downloader import download_arxiv_paper

# Import the tokenizer (this is key!)
from tokenizer import get_tokenizer

# Import the chunked analyzer and base analyzers
from analyser.base_analyser import BaseAnalyser
from analyser.chunked_analyser import ChunkedAnalyser, create_chunked_analyser
from analyser.deepseek_analyser import DeepSeekAnalyser
from analyser.gemini_analyser import GeminiAnalyser

# Import prompt templates
from analyser import prompt_templates
from analyser import prompt_templates_robotics

# Import the db_integration module

# Import database integration
from analyser.db_integration import (
    analyze_and_store, 
    store_paper_from_md_file, 
    get_or_create_model, 
    store_extraction_results
)

# Set up environment variables if not already set
from dotenv import load_dotenv
load_dotenv()

print(" Libraries imported successfully")

## Step 0: arXiv Paper Download (Optional)

Download papers directly from arXiv using their paper IDs. This step is optional - you can skip it if you already have PDF files.

In [ ]:
# arXiv Download Configuration
ARXIV_CONFIG = {
    # Set to True to download from arXiv, False to use existing PDF
    'download_from_arxiv': True,
    
    # arXiv paper ID (e.g., '2101.12345' or '2101.12345v1')
    'arxiv_id': '2402.07827',
    
    # Directory to save downloaded papers
    'download_dir': 'downloaded_papers',
    
    # Existing PDF path (used if download_from_arxiv is False)
    'existing_pdf': 'sample_data/samplePDF.pdf'
}

print(f" arXiv Download Configuration:")
print(f"  - Download from arXiv: {ARXIV_CONFIG['download_from_arxiv']}")
print(f"  - arXiv ID: {ARXIV_CONFIG['arxiv_id']}")
print(f"  - Download directory: {ARXIV_CONFIG['download_dir']}")
print(f"  - Existing PDF: {ARXIV_CONFIG['existing_pdf']}")

In [ ]:
def download_arxiv_paper_for_pipeline(config):
    """Download paper from arXiv or use existing PDF."""
    
    print(f"\n=== Step 0: Paper Acquisition ===")
    
    if config['download_from_arxiv']:
        print(f" Downloading paper from arXiv: {config['arxiv_id']}")
        
        # Create download directory
        os.makedirs(config['download_dir'], exist_ok=True)
        
        # Download the paper
        start_time = time.time()
        pdf_path, metadata = download_arxiv_paper(config['arxiv_id'], config['download_dir'])
        elapsed_time = time.time() - start_time
        
        if pdf_path:
            print(f" Paper downloaded successfully in {elapsed_time:.2f} seconds")
            print(f"  - PDF path: {pdf_path}")
            print(f"  - Title: {metadata['title'][:100]}...")
            print(f"  - Authors: {', '.join(metadata['authors'][:3])}{'...' if len(metadata['authors']) > 3 else ''}")
            print(f"  - Published: {metadata['published'].strftime('%Y-%m-%d')}")
            print(f"  - Categories: {', '.join(metadata['categories'])}")
            
            # Get file size
            file_size_mb = os.path.getsize(pdf_path) / (1024 * 1024)
            print(f"  - File size: {file_size_mb:.2f} MB")
            
            return pdf_path, metadata
        else:
            print(f" Failed to download paper from arXiv after {elapsed_time:.2f} seconds")
            print(f"  - Check if arXiv ID '{config['arxiv_id']}' is valid")
            print(f"  - Check internet connection")
            return None, None
    else:
        print(f" Using existing PDF file: {config['existing_pdf']}")
        
        # Check if existing PDF exists
        if os.path.exists(config['existing_pdf']):
            file_size_mb = os.path.getsize(config['existing_pdf']) / (1024 * 1024)
            print(f" PDF file found")
            print(f"  - Path: {config['existing_pdf']}")
            print(f"  - Size: {file_size_mb:.2f} MB")
            
            # Create minimal metadata for existing file
            metadata = {
                'title': f"Local PDF: {Path(config['existing_pdf']).stem}",
                'authors': ['Unknown'],
                'published': None,
                'categories': ['Unknown'],
                'summary': 'Local PDF file'
            }
            
            return config['existing_pdf'], metadata
        else:
            print(f" PDF file not found: {config['existing_pdf']}")
            print(f"\n Available sample files:")
            sample_dir = 'sample_data'
            if os.path.exists(sample_dir):
                for file in os.listdir(sample_dir):
                    if file.endswith('.pdf'):
                        full_path = os.path.join(sample_dir, file)
                        size_mb = os.path.getsize(full_path) / (1024 * 1024)
                        print(f"  - {file} ({size_mb:.2f} MB)")
            else:
                print(f"  - No sample_data directory found")
            return None, None

# Download or locate the paper
pdf_file_path, paper_metadata = download_arxiv_paper_for_pipeline(ARXIV_CONFIG)

if pdf_file_path:
    print(f"\n Paper ready for processing: {pdf_file_path}")
else:
    print(f"\n️ No paper available - please check configuration")

## Enhanced OCR Configuration and Functions

Configure Azure OCR API for PDF processing with detailed image annotations.

### Features:
- **Detailed Image Annotations**: Comprehensive descriptions of visual content only
- **Data Extraction**: Specific values and information from charts and graphs
- **Visual Context**: Understanding of how images relate to document content
- **Focused Processing**: Only processes images, graphs, tables, and figures

In [ ]:
# Import enhanced OCR modules
from ocr_models import ImageAnnotation, ImageType, DataVisualizationType
from ocr_analysis import EnhancedOCRProcessor, process_pdf_with_enhanced_ocr

# Add standard OCR processing function for comparison
def process_pdf_with_ocr(pdf_path, output_dir, api_key=None, api_endpoint=None, timeout=120, verbose=True):
    """Standard OCR processing without image annotations (for comparison)."""
    
    if not api_key:
        api_key = os.getenv('AZURE_OCR_API_KEY', 'your_azure_ocr_api_key_here')
    if not api_endpoint:
        api_endpoint = os.getenv('AZURE_OCR_ENDPOINT', 'https://mistral-ocr-2503-zxbfx.eastus.models.ai.azure.com/v1/ocr')
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    pdf_name = Path(pdf_path).stem


    
    # Encode PDF
    try:
        with open(pdf_path, "rb") as pdf_file:
            file_content = pdf_file.read()
            base64_pdf = base64.b64encode(file_content).decode('utf-8')
    except Exception as e:
        if verbose:
            print(f" Error encoding PDF: {e}")
        return None, None
    
    # Standard OCR payload (no annotations)
    payload = {
        "model": "mistral-ocr-latest",
        "document": {
            "document_url": f"data:application/pdf;base64,{base64_pdf}",
            "type": "document_url"
        }
    }
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }
    
    try:
        if verbose:
            print("  - Sending standard OCR request to Azure API...")
        
        start_time = time.time()
        response = requests.post(
            api_endpoint,
            headers=headers,
            json=payload,
            timeout=timeout
        )
        elapsed_time = time.time() - start_time
        
        if verbose:
            print(f"  - API response received in {elapsed_time:.2f}s (Status: {response.status_code})")
        
        if response.status_code == 200:
            result = response.json()
            
            if result is None:
                if verbose:
                    print(" OCR API returned None response")
                return None, None
            
            # Extract text content
            extracted_text = ""
            if 'pages' in result and result['pages']:
                for page_idx, page in enumerate(result['pages']):
                    if page and 'markdown' in page:
                        extracted_text += f"\n\n--- Page {page_idx + 1} ---\n\n"
                        extracted_text += page['markdown'] + "\n\n"
            
            # Save files
            md_path = os.path.join(output_dir, f"{pdf_name}.md")
            json_path = os.path.join(output_dir, f"{pdf_name}_ocr_response.json")
            
            with open(md_path, 'w', encoding='utf-8') as f:
                f.write(extracted_text)
            
            with open(json_path, 'w', encoding='utf-8') as f:
                json.dump(result, f, indent=2, ensure_ascii=False)
            
            if verbose:
                print(f" Standard OCR processing complete:")
                print(f"  - Markdown: {md_path}")
                print(f"  - JSON: {json_path}")
                print(f"  - Text length: {len(extracted_text):,} characters")
            
            return md_path, json_path
        else:
            if verbose:
                print(f" OCR API error: {response.status_code}")
                print(f"  Response: {response.text[:200]}...")
            return None, None
            
    except Exception as e:
        if verbose:
            print(f" OCR processing error: {e}")
        return None, None

# OCR Configuration - Enhanced for Images Only
ENHANCED_OCR_CONFIG = {
    'api_key': os.getenv('AZURE_OCR_API_KEY', 'your_azure_ocr_api_key_here'),
    'api_endpoint': os.getenv('AZURE_OCR_ENDPOINT', 'https://mistral-ocr-2503-zxbfx.eastus.models.ai.azure.com/v1/ocr'),
    'timeout': 180,  # Increased timeout for annotation processing
    'max_annotation_pages': 8,  # API limit for detailed annotations
    'include_images': False,  # Don't include base64 images when using detailed annotations
    'use_annotations': False,  # SET TO FALSE IF ENDPOINT DOESN'T SUPPORT ANNOTATIONS
    'verbose': True
}

def run_enhanced_ocr_processing(config):
    """Run enhanced OCR processing with optional detailed image annotations."""
    
    print(f"\n=== Step 1: Enhanced OCR Processing ===")
    
    if not config['process_pdf']:
        print(" OCR processing disabled in config")
        return None, None, None
    
    pdf_path = config['pdf_path']
    
    # Check if PDF exists
    if not os.path.exists(pdf_path):
        print(f" PDF file not found: {pdf_path}")
        return None, None, None
    
    # Get file info
    file_size_mb = os.path.getsize(pdf_path) / (1024 * 1024)
    print(f" Input PDF: {pdf_path} ({file_size_mb:.2f} MB)")
    
    ocr_mode = "Enhanced OCR with bbox annotations" if ENHANCED_OCR_CONFIG['use_annotations'] else "Standard OCR"
    print(f" Processing mode: {ocr_mode}")
    if ENHANCED_OCR_CONFIG['use_annotations']:
        print(f"  - Primary: Enhanced OCR with detailed image annotations")
        print(f"  - Fallback: Standard OCR if enhanced fails")
    else:
        print(f"  - Using: Standard OCR (annotations disabled in config)")
    
    # Run OCR processing
    start_time = time.time()
    md_path, json_path, annotations = process_pdf_with_enhanced_ocr(
        pdf_path=pdf_path,
        output_dir=config['output_dir'],
        api_key=ENHANCED_OCR_CONFIG['api_key'],
        api_endpoint=ENHANCED_OCR_CONFIG['api_endpoint'],
        max_pages=ENHANCED_OCR_CONFIG['max_annotation_pages'],
        include_images=ENHANCED_OCR_CONFIG['include_images'],
        use_annotations=ENHANCED_OCR_CONFIG['use_annotations'],  # Pass the flag
        verbose=ENHANCED_OCR_CONFIG['verbose']
    )
    elapsed_time = time.time() - start_time
    
    if md_path and json_path:
        if annotations and ENHANCED_OCR_CONFIG['use_annotations']:
            print(f"\n Enhanced OCR with annotations completed in {elapsed_time:.2f} seconds")
            print(f"  - Markdown file: {md_path}")
            print(f"  - JSON response: {json_path}")
            print(f"  - Total image annotations: {annotations['total_annotations']}")
            
            # Display annotation summary
            if annotations['total_annotations'] > 0:
                print(f"\n Image Annotation Summary:")
                print(f"\n Annotation Types:")
                for img_type, count in annotations.get('annotation_types', {}).items():
                    print(f"  - {img_type.title()}: {count}")
                
                print(f"\n By Page:")
                for page_num, page_annotations in annotations['annotations_by_page'].items():
                    print(f"  Page {page_num}: {len(page_annotations)} annotations")
                    for i, annotation in enumerate(page_annotations[:2]):  # Show first 2 per page
                        img_type = annotation.get('image_type', 'Unknown')
                        title = annotation.get('title', 'Untitled')[:40]
                        print(f"    {i+1}. {img_type}: {title}...")
                    if len(page_annotations) > 2:
                        print(f"    ... and {len(page_annotations) - 2} more")
        else:
            ocr_type = "Standard OCR" if not ENHANCED_OCR_CONFIG['use_annotations'] else "Standard OCR fallback"
            print(f"\n {ocr_type} completed in {elapsed_time:.2f} seconds")
            print(f"  - Markdown file: {md_path}")
            print(f"  - JSON response: {json_path}")
            if not ENHANCED_OCR_CONFIG['use_annotations']:
                print(f"  - Note: Enhanced annotations disabled in config")
            else:
                print(f"  - Note: Enhanced annotations not available (used fallback)")
        
        # Check the generated markdown
        with open(md_path, 'r', encoding='utf-8') as f:
            content = f.read()
        print(f"  - Generated text: {len(content):,} characters")
        print(f"  - Preview: {content[:300]}...")
        
        return md_path, json_path, annotations
    else:
        print(f" OCR processing failed after {elapsed_time:.2f} seconds")
        return None, None, None


    """Run enhanced OCR processing with optional detailed image annotations."""
    
    print(f"\n=== Step 1: Enhanced OCR Processing ===")
    
    if not config['process_pdf']:
        print(" OCR processing disabled in config")
        return None, None, None
    
    pdf_path = config['pdf_path']
    
    # Check if PDF exists
    if not os.path.exists(pdf_path):
        print(f" PDF file not found: {pdf_path}")
        return None, None, None
    
    # Get file info
    file_size_mb = os.path.getsize(pdf_path) / (1024 * 1024)
    print(f" Input PDF: {pdf_path} ({file_size_mb:.2f} MB)")
    
    ocr_mode = "Enhanced OCR with bbox annotations" if ENHANCED_OCR_CONFIG['use_annotations'] else "Standard OCR"
    print(f" Processing mode: {ocr_mode}")
    if ENHANCED_OCR_CONFIG['use_annotations']:
        print(f"  - Primary: Enhanced OCR with detailed image annotations")
        print(f"  - Fallback: Standard OCR if enhanced fails")
    else:
        print(f"  - Using: Standard OCR (annotations disabled in config)")
    
    # Run OCR processing
    start_time = time.time()
    md_path, json_path, annotations = process_pdf_with_enhanced_ocr(
        pdf_path=pdf_path,
        output_dir=config['output_dir'],
        api_key=ENHANCED_OCR_CONFIG['api_key'],
        api_endpoint=ENHANCED_OCR_CONFIG['api_endpoint'],
        max_pages=ENHANCED_OCR_CONFIG['max_annotation_pages'],
        include_images=ENHANCED_OCR_CONFIG['include_images'],
        use_annotations=ENHANCED_OCR_CONFIG['use_annotations'],  # Pass the flag
        verbose=ENHANCED_OCR_CONFIG['verbose']
    )
    elapsed_time = time.time() - start_time
    
    if md_path and json_path:
        if annotations and ENHANCED_OCR_CONFIG['use_annotations']:
            print(f"\n Enhanced OCR with annotations completed in {elapsed_time:.2f} seconds")
            print(f"  - Markdown file: {md_path}")
            print(f"  - JSON response: {json_path}")
            print(f"  - Total image annotations: {annotations['total_annotations']}")
            
            # Display annotation summary
            if annotations['total_annotations'] > 0:
                print(f"\n Image Annotation Summary:")
                print(f"\n Annotation Types:")
                for img_type, count in annotations.get('annotation_types', {}).items():
                    print(f"  - {img_type.title()}: {count}")
                
                print(f"\n By Page:")
                for page_num, page_annotations in annotations['annotations_by_page'].items():
                    print(f"  Page {page_num}: {len(page_annotations)} annotations")
                    for i, annotation in enumerate(page_annotations[:2]):  # Show first 2 per page
                        img_type = annotation.get('image_type', 'Unknown')
                        title = annotation.get('title', 'Untitled')[:40]
                        print(f"    {i+1}. {img_type}: {title}...")
                    if len(page_annotations) > 2:
                        print(f"    ... and {len(page_annotations) - 2} more")
        else:
            ocr_type = "Standard OCR" if not ENHANCED_OCR_CONFIG['use_annotations'] else "Standard OCR fallback"
            print(f"\n {ocr_type} completed in {elapsed_time:.2f} seconds")
            print(f"  - Markdown file: {md_path}")
            print(f"  - JSON response: {json_path}")
            if not ENHANCED_OCR_CONFIG['use_annotations']:
                print(f"  - Note: Enhanced annotations disabled in config")
            else:
                print(f"  - Note: Enhanced annotations not available (used fallback)")
        
        # Check the generated markdown
        with open(md_path, 'r', encoding='utf-8') as f:
            content = f.read()
        print(f"  - Generated text: {len(content):,} characters")
        print(f"  - Preview: {content[:300]}...")
        
        return md_path, json_path, annotations
    else:
        print(f" OCR processing failed after {elapsed_time:.2f} seconds")
        return None, None, None

# Configuration for complete pipeline
CONFIG = {
    # Input files - now uses the downloaded/selected PDF
    'pdf_path': pdf_file_path if pdf_file_path else 'sample_data/samplePDF.pdf',
    'paper_path': None,  # Will be set after OCR processing
    'output_dir': f'results/{ARXIV_CONFIG["arxiv_id"].replace("/", "_")}' if ARXIV_CONFIG['download_from_arxiv'] else 'results/complete_pipeline',
    
    # Paper metadata from arXiv or local file
    'paper_metadata': paper_metadata,
    
    # Analysis settings
    'provider': 'deepseek',  # 'deepseek', 'gemini', 
    'max_tokens': 6000,  # Maximum tokens per chunk
    'overlap_tokens': 500,  # Overlap between chunks
    'progressive': True,  # Use progressive chunking
    'exclude_sections': ['References'],  # Sections to exclude
    'use_robotics_prompts': False,  # Toggle between standard and robotics prompts
    
    # API tokens
    'deepseek_token': os.getenv('DEEPSEEK_API_TOKEN'),
    'gemini_token': os.getenv('GEMINI_API_KEY'),
    
    
    # Database options
    'save_to_db': True,  # Whether to save results to database
    'arxiv_id': ARXIV_CONFIG['arxiv_id'] if ARXIV_CONFIG['download_from_arxiv'] else 'pipeline-test-001',
    'title': paper_metadata['title'] if paper_metadata else 'Complete Pipeline Test Paper',
    
    # OCR options
    'process_pdf': True,  # Whether to run OCR on PDF first
    'save_images': True   # Whether to save extracted images
}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print(f" Complete Pipeline Configuration:")
print(f"  prompt_mode: {'Robotics' if CONFIG.get('use_robotics_prompts', False) else 'Standard'}")
for key, value in CONFIG.items():
    if 'token' in key.lower():
        print(f"  {key}: {' Set' if value else ' Not set'}")
    elif key == 'paper_metadata':
        print(f"  {key}: {' Available' if value else ' Not available'}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Run enhanced OCR processing
#enhanced_md_file, enhanced_json_file, enhanced_annotations = run_enhanced_ocr_processing(CONFIG)

# Update config with the generated markdown file
if enhanced_md_file:
    CONFIG['paper_path'] = enhanced_md_file
    print(f"\n Updated paper_path to: {CONFIG['paper_path']}")
    
    # Analyze the annotations
    #analyze_ocr_annotations(enhanced_annotations)
    
    # Show sample annotation details
    if enhanced_annotations and enhanced_annotations.get('all_annotations'):
        print(f"\n Sample Detailed Annotation:")
        sample_annotation = enhanced_annotations['all_annotations'][0]
        print(f"  Type: {sample_annotation.get('image_type', 'N/A')}")
        print(f"  Title: {sample_annotation.get('title', 'N/A')}")
        print(f"  Description: {sample_annotation.get('detailed_description', 'N/A')[:200]}...")
        if sample_annotation.get('data_content'):
            print(f"  Data: {sample_annotation['data_content'][:150]}...")
        if sample_annotation.get('key_insights'):
            print(f"  Insights: {sample_annotation['key_insights'][:150]}...")
else:
    print(f"\n️ Enhanced OCR processing failed - will need existing markdown file for next steps")

## Step 1: OCR Processing (PDF → Markdown)

Convert PDF file to markdown text using Azure OCR API.

In [ ]:
def run_ocr_processing(config):
    """Run OCR processing on the input PDF file."""
    
    print(f"\n=== Step 1: OCR Processing ===")
    
    if not config['process_pdf']:
        print(" OCR processing disabled in config")
        return None, None
    
    pdf_path = config['pdf_path']
    
    # Check if PDF exists
    if not os.path.exists(pdf_path):
        print(f" PDF file not found: {pdf_path}")
        print("\n Available sample files:")
        sample_dir = 'sample_data'
        if os.path.exists(sample_dir):
            for file in os.listdir(sample_dir):
                if file.endswith('.pdf'):
                    full_path = os.path.join(sample_dir, file)
                    size_kb = os.path.getsize(full_path) / 1024
                    print(f"  - {file} ({size_kb:.1f} KB)")
        return None, None
    
    # Get file info
    file_size_kb = os.path.getsize(pdf_path) / 1024
    print(f" Input PDF: {pdf_path} ({file_size_kb:.2f} KB)")
    
    # Process with OCR
    start_time = time.time()
    md_path, json_path = process_pdf_with_ocr(pdf_path, config['output_dir'])
    elapsed_time = time.time() - start_time
    
    if md_path and json_path:
        print(f"\n OCR processing completed in {elapsed_time:.2f} seconds")
        print(f"  - Markdown file: {md_path}")
        print(f"  - JSON response: {json_path}")
        
        # Check the generated markdown
        with open(md_path, 'r', encoding='utf-8') as f:
            content = f.read()
        print(f"  - Generated text: {len(content):,} characters")
        print(f"  - Preview: {content[:200]}...")
        
        return md_path, json_path
    else:
        print(f" OCR processing failed after {elapsed_time:.2f} seconds")
        return None, None

# Run OCR processing
#md_file, json_file = run_ocr_processing(CONFIG)
print("Skipping OCR processing for now...")
md_file, json_file = None, None

# Update config with the generated markdown file
if md_file:
    CONFIG['paper_path'] = md_file
    print(f"\n Updated paper_path to: {CONFIG['paper_path']}")
else:
    print(f"\n️ OCR processing failed - will need existing markdown file for next steps")

## Step 2: Token Analysis

Analyze the markdown file to determine if chunking is needed.

In [ ]:
def analyze_document_tokens(paper_path, provider='deepseek', max_tokens=8192):
    """Analyze document tokens to determine chunking needs."""
    
    print(f"\n=== Step 2: Token Analysis ===")
    
    if not paper_path or not os.path.exists(paper_path):
        print(f" Paper not found: {paper_path}")
        return None, None
    
    # Get file info
    file_size_kb = os.path.getsize(paper_path) / 1024
    print(f" Document: {paper_path} ({file_size_kb:.2f} KB)")
    
    # Count tokens using the tokenizer
    tokenizer = get_tokenizer(provider, max_tokens=max_tokens)
    if not tokenizer:
        print(f" Could not create tokenizer for provider: {provider}")
        return None, None
    
    try:
        with open(paper_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        token_count = tokenizer.count_tokens(content)
        print(f" Token Analysis:")
        print(f"  - Content length: {len(content):,} characters")
        print(f"  - Token count: {token_count:,} tokens")
        print(f"  - Tokenizer max: {tokenizer.max_tokens:,} tokens")
        
        # Calculate effective limits
        prompt_buffer = 1000  # Tokens for system prompt
        response_buffer = 1000  # Tokens for response
        effective_limit = tokenizer.max_tokens - prompt_buffer - response_buffer
        
        needs_chunking = token_count > effective_limit
        
        print(f"  - Effective limit: {effective_limit:,} tokens")
        print(f"  - Chunking needed: {'Yes' if needs_chunking else 'No'}")
        
        if needs_chunking:
            estimated_chunks = (token_count // CONFIG['max_tokens']) + 1
            print(f"  - Estimated chunks: {estimated_chunks}")
        
        return token_count, needs_chunking
        
    except Exception as e:
        print(f" Error analyzing tokens: {e}")
        return None, None

# Analyze document tokens
if CONFIG['paper_path']:
    token_count, needs_chunking = analyze_document_tokens(CONFIG['paper_path'], CONFIG['provider'])
else:
    print("️ No paper path available for token analysis")
    token_count, needs_chunking = None, None

## Step 3: Complete Pipeline Analysis with Database Storage

Run the full analysis pipeline with intelligent chunking and database storage.

In [ ]:
def run_complete_pipeline_analysis(config):
    """Run the complete analysis pipeline with database storage."""
    
    print(f"\n=== Step 3: Complete Pipeline Analysis ===")
    
    paper_path = config['paper_path']
    if not paper_path or not os.path.exists(paper_path):
        print(f" Paper not available for analysis: {paper_path}")
        return None, None, None
    
    try:
        # Select prompt template based on configuration
        prompt_template = prompt_templates_robotics if config.get('use_robotics_prompts', False) else prompt_templates
        
        # Initialize base analyzer
        print(f" Initializing {config['provider']} analyzer...")
        
        if config['provider'].lower() == 'deepseek':
            base_analyser = DeepSeekAnalyser(prompt_template=prompt_template)
            model_name = base_analyser.model
        elif config['provider'].lower() == 'gemini':
            api_key = config['gemini_token']
            if not api_key:
                print(" Gemini API key not found in environment variable GEMINI_API_KEY")
                return None, None, None
            base_analyser = GeminiAnalyser(api_key=api_key, prompt_template=prompt_template)
            model_name = base_analyser.model_name
        else:
            raise ValueError(f"Unsupported provider: {config['provider']}")
        
        print(f" Created {config['provider']} analyzer with model: {model_name}")
        
        # Determine analysis approach based on token count
        if needs_chunking:
            print(f"\n Using chunked analysis (document is large)")
            
            # Create chunked analyser
            chunked_analyser = create_chunked_analyser(
                base_analyser=base_analyser,
                provider=config['provider'],
                max_tokens=config['max_tokens'],
                progressive=config['progressive'],
                template_module=prompt_template
            )
            
            print(f" Chunked analyzer configured:")
            print(f"  - Max tokens per chunk: {config['max_tokens']:,}")
            print(f"  - Progressive mode: {config['progressive']}")
            print(f"  - Overlap tokens: {config['overlap_tokens']}")
            
            # Set excluded sections
            if hasattr(chunked_analyser, 'exclude_sections') and config['exclude_sections']:
                chunked_analyser.exclude_sections = config['exclude_sections']
                print(f"  - Excluding sections: {', '.join(config['exclude_sections'])}")
            
            # Run chunked analysis
            start_time = time.time()
            text_file, json_file = chunked_analyser.analyze_publication(paper_path)
            elapsed_time = time.time() - start_time
            
        else:
            print(f"\n Using direct analysis (document fits in context)")
            
            # Run direct analysis
            start_time = time.time()
            response = base_analyser.analyze_from_md_file(paper_path)
            elapsed_time = time.time() - start_time
            
            # Save response to files
            pdf_name = Path(paper_path).stem
            text_file = os.path.join(config['output_dir'], f"{pdf_name}_direct_response.txt")
            json_file = os.path.join(config['output_dir'], f"{pdf_name}_direct_response.json")
            
            with open(text_file, 'w', encoding='utf-8') as f:
                f.write(str(response))
            
            # Try to parse as JSON
            try:
                if isinstance(response, str):
                    response_json = json.loads(response)
                else:
                    response_json = response
                with open(json_file, 'w', encoding='utf-8') as f:
                    json.dump(response_json, f, indent=2)
            except:
                json_file = None
                print("  - Could not parse response as JSON")
        
        # Report analysis results
        if text_file and (json_file or not needs_chunking):
            print(f"\n Analysis completed in {elapsed_time:.2f} seconds")
            print(f"  - Text response: {text_file}")
            if json_file:
                print(f"  - JSON response: {json_file}")
            
            # Parse and display results
            response_data = None
            if json_file and os.path.exists(json_file):
                try:
                    with open(json_file, 'r', encoding='utf-8') as f:
                        response_data = json.load(f)
                    
                    if isinstance(response_data, list):
                        print(f"\n Extracted {len(response_data)} model variants:")
                        for i, model in enumerate(response_data[:3]):  # Show first 3
                            if isinstance(model, dict) and "model_name" in model:
                                model_name_value = model["model_name"].get("value", "Unknown")
                                fields_count = sum(1 for field in model if model[field].get("value") not in ["n/a", None, ""])
                                total_fields = len(model)
                                print(f"  {i+1}. {model_name_value}: {fields_count}/{total_fields} fields")
                        if len(response_data) > 3:
                            print(f"  ... and {len(response_data) - 3} more model variants")
                except Exception as e:
                    print(f"  - Could not parse JSON response: {e}")
            
            # Save to database if requested
            db_info = None
            if config['save_to_db'] and response_data:
                print(f"\n️ Saving results to database...")
                try:
                    # Store the paper
                    paper_id = store_paper_from_md_file(
                        paper_path, 
                        arxiv_id=config.get('arxiv_id'), 
                        title=config.get('title')
                    )
                    
                    if paper_id:
                        print(f" Paper stored with ID: {paper_id}")
                        
                        # Get or create model record
                        model_id = get_or_create_model(
                            model_name=model_name,
                            provider=config['provider'].title(),
                            context_size=config.get('max_tokens')
                        )
                        
                        if model_id:
                            print(f" Model stored with ID: {model_id}")
                            
                            # Store extraction results
                            run_ids = store_extraction_results(
                                paper_id=paper_id,
                                model_id=model_id,
                                response=response_data,
                                temperature=getattr(base_analyser, 'temperature', None)
                            )
                            
                            if run_ids:
                                print(f" Stored {len(run_ids)} extraction runs")
                                db_info = {
                                    'paper_id': paper_id,
                                    'model_id': model_id,
                                    'run_ids': run_ids
                                }
                                print(f" Database storage completed successfully!")
                            else:
                                print(" Failed to store extraction results")
                        else:
                            print(" Failed to store model")
                    else:
                        print(" Failed to store paper")
                        
                except Exception as e:
                    print(f" Database error: {e}")
                    import traceback
                    traceback.print_exc()
            
            return text_file, json_file, db_info
            
        else:
            print(f" Analysis failed after {elapsed_time:.2f} seconds")
            return None, None, None
            
    except Exception as e:
        print(f" Pipeline error: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None

# Run complete pipeline analysis
if True or (CONFIG['paper_path'] and (CONFIG['deepseek_token'] or CONFIG['gemini_token'])):
    #analysis_text, analysis_json, db_results = run_complete_pipeline_analysis(CONFIG)
    print("️ Skipping analysis step - not implemented in this example")
else:
    print("️ Cannot run analysis: Missing paper file or API tokens")
    analysis_text, analysis_json, db_results = None, None, None

## Step 4: Examine Results

Review the complete pipeline results including database storage.

In [ ]:
def examine_pipeline_results(db_results, config):
    """Examine the complete pipeline results."""
    
    print(f"\n=== Step 4: Pipeline Results Summary ===")
    
    # OCR Results
    if config.get('paper_path'):
        print(f"\n OCR Processing:")
        print(f"  - Input PDF: {config['pdf_path']}")
        print(f"  - Generated Markdown: {config['paper_path']}")
        if os.path.exists(config['paper_path']):
            with open(config['paper_path'], 'r', encoding='utf-8') as f:
                content = f.read()
            print(f"  - Content length: {len(content):,} characters")
    
    # Token Analysis
    print(f"\n Token Analysis:")
    if token_count:
        print(f"  - Token count: {token_count:,}")
        print(f"  - Chunking needed: {needs_chunking}")
        print(f"  - Provider: {config['provider']}")
    else:
        print(f"  - Token analysis not completed")
    
    # Analysis Results
    print(f"\n LLM Analysis:")
    if analysis_text and analysis_json:
        print(f"  - Text response: {analysis_text}")
        print(f"  - JSON response: {analysis_json}")
        print(f"  - Analysis method: {'Chunked' if needs_chunking else 'Direct'}")
    else:
        print(f"  - Analysis not completed")
    
    # Database Results
    print(f"\n️ Database Storage:")
    if db_results:
        print(f"  - Paper ID: {db_results['paper_id']}")
        print(f"  - Model ID: {db_results['model_id']}")
        print(f"  - Extraction runs: {len(db_results['run_ids'])}")
        print(f"  - Run IDs: {db_results['run_ids']}")
        
        # Quick database query
        try:
            from db.paper_dao import PaperDAO
            from db.extraction_dao import ExtractionDAO
            
            paper = PaperDAO.get_paper_by_id(db_results['paper_id'])
            if paper:
                print(f"  - Paper title: {paper.get('title', 'N/A')}")
                print(f"  - ArXiv ID: {paper.get('arxiv_id', 'N/A')}")
            
            # Count extracted fields
            total_fields = 0
            for run_id in db_results['run_ids']:
                fields = ExtractionDAO.get_extracted_fields(run_id)
                if fields:
                    total_fields += len(fields)
            print(f"  - Total extracted fields: {total_fields}")
            
        except Exception as e:
            print(f"  - Database query error: {e}")
    else:
        print(f"  - No database results available")
    
    # Output Files Summary
    print(f"\n Output Files:")
    output_dir = config['output_dir']
    if os.path.exists(output_dir):
        files = [f for f in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, f))]
        print(f"  - Output directory: {output_dir}")
        print(f"  - Files created: {len(files)}")
        for file in sorted(files)[:10]:  # Show first 10 files
            file_path = os.path.join(output_dir, file)
            size_kb = os.path.getsize(file_path) / 1024
            print(f"    - {file} ({size_kb:.1f} KB)")
        if len(files) > 10:
            print(f"    ... and {len(files) - 10} more files")
    
    # Pipeline Status
    print(f"\n Pipeline Status:")
    ocr_status = "" if config.get('paper_path') else ""
    token_status = "" if token_count else ""
    analysis_status = "" if analysis_json else ""
    db_status = "" if db_results else ""
    
    print(f"  - OCR Processing: {ocr_status}")
    print(f"  - Token Analysis: {token_status}")
    print(f"  - LLM Analysis: {analysis_status}")
    print(f"  - Database Storage: {db_status}")
    
    pipeline_complete = all([
        config.get('paper_path'),
        token_count,
        analysis_json,
        db_results if config['save_to_db'] else True
    ])
    
    if pipeline_complete:
        print(f"\n Complete pipeline executed successfully!")
    else:
        print(f"\n️ Pipeline partially completed - check individual steps above")

# Examine pipeline results
examine_pipeline_results(db_results, CONFIG)

## Database Management and Next Steps

Commands for managing the database and analyzing results.

In [ ]:
def show_database_management_commands(config, db_results):
    """Show available database management commands."""
    
    print(f"\n️ Database Management Commands:")
    print()
    print("1. Setup database (run once):")
    print("   python manage_db.py setup")
    print()
    print("2. List all papers in database:")
    print("   python manage_db.py list-papers --verbose")
    print()
    if db_results and config.get('arxiv_id'):
        print(f"3. Calculate consensus for this paper:")
        print(f"   python manage_db.py calculate-consensus --arxiv-id '{config['arxiv_id']}' --method weighted")
        print()
        print(f"4. View extraction runs for this paper:")
        print(f"   python manage_db.py list-extractions --paper-id {db_results['paper_id']}")
        print()
    
    print("5. Export all consensus results:")
    print("   python manage_db.py export-consensus --output consensus_results.json")
    print()
    print("6. Backup database:")
    print("   python manage_db.py backup --output backup.sql")
    print()
    
    print(f"\n Re-run Pipeline Options:")
    print()
    print("To process a different PDF:")
    print(f"  1. Update CONFIG['pdf_path'] to your new PDF file")
    print(f"  2. Update CONFIG['arxiv_id'] to a unique identifier")
    print(f"  3. Run all cells from 'Step 1: OCR Processing' onwards")
    print()
    print("To use a different LLM provider:")
    print(f"  1. Update CONFIG['provider'] to 'deepseek', 'gemini'")
    print(f"  2. Ensure the corresponding API key is set in environment")
    print(f"  3. Re-run from 'Step 3: Complete Pipeline Analysis'")
    print()
    print("To analyze an existing markdown file:")
    print(f"  1. Set CONFIG['process_pdf'] = False")
    print(f"  2. Set CONFIG['paper_path'] to your markdown file")
    print(f"  3. Run from 'Step 2: Token Analysis' onwards")
    print()
    print(f"\n Analysis Results Location:")
    if config.get('output_dir'):
        print(f"  - Output directory: {config['output_dir']}")
        print(f"  - Contains: PDF→MD conversion, analysis results, extracted images")
    print()
    print(f" Tips:")
    print(f"  - Adjust CONFIG['max_tokens'] for different chunk sizes")
    print(f"  - Set CONFIG['progressive'] = False for simple chunking")
    print(f"  - Use CONFIG['exclude_sections'] to skip certain parts")
    print(f"  - Check logs for detailed processing information")

# Show database management commands
show_database_management_commands(CONFIG, db_results)

print(f"\n Complete Pipeline Summary:")
print(f"  Input: PDF file → OCR → Markdown → Token Analysis → LLM Analysis → Database")
print(f"  Status: Pipeline setup complete and ready to use!")
print(f"  Next: Run the cells above to process your PDF files")

## Integrated Pipeline Function

This function combines all pipeline steps into a single, comprehensive workflow that:
1. Downloads papers from arXiv (or uses existing PDFs)
2. Processes PDFs with OCR to extract text and images
3. Analyzes token count and applies chunking if needed
4. Runs LLM analysis with multiple providers and runs
5. Stores all results in the database
6. Provides detailed progress reporting and error handling

In [ ]:
def run_complete_pipeline(
    # Paper source configuration
    arxiv_id: Optional[str] = None,
    pdf_path: Optional[str] = None,
    download_dir: str = "downloaded_papers",
    
    # Enhanced OCR configuration
    run_ocr: bool = True,
    use_enhanced_ocr: bool = True,  # Enhanced image annotations
    use_annotations: bool = False,  # NEW: Control annotation usage
    ocr_output_dir: Optional[str] = None,
    max_annotation_pages: int = 8,
    force_reprocess: bool = False,  # NEW: Force redownload/reprocess even if MD exists
    
    # Analysis configuration
    run_analysis: bool = True,
    deepseek_runs: int = 1,
    gemini_runs: int = 0,
    
    # Advanced settings
    temperature: float = 0.3,
    max_chunk_size: int = 15000,
    chunk_overlap: int = 200,
    use_robotics_prompts: bool = False,  # NEW: Toggle between standard and robotics prompts
    
    # Progress reporting
    verbose: bool = True
) -> dict:
    """
    Run the complete PDF-to-Database pipeline with enhanced image annotations.
    
    Enhanced OCR focuses specifically on visual elements:
    - Detailed analysis of images, graphs, charts, tables
    - Data extraction from visualizations
    - Technical details and scientific information
    - Context relevance for each visual element
    
    Args:
        use_enhanced_ocr: Whether to use enhanced OCR with detailed image annotations
        max_annotation_pages: Maximum pages to process for detailed annotations (API limit: 8)
        force_reprocess: If False, check for existing MD files before downloading/OCR processing
    """
    
    pipeline_results = {
        'success': False,
        'paper_id': None,
        'pdf_path': None,
        'md_path': None,
        'ocr_results': None,
        'image_annotations': None,  # Focused on images only
        'analysis_results': {
            'deepseek': [],
            'gemini': [],
        },
        'images_stored': 0,
        'image_annotations_count': 0,  # Count of detailed image annotations
        'total_extractions': 0,
        'processing_time': 0,
        'errors': [],
        'skipped_download': False,  # NEW: Track if download was skipped
        'skipped_ocr': False,       # NEW: Track if OCR was skipped
        'existing_md_used': None   # NEW: Path to existing MD file if used
    }
    
    import time
    import os
    from pathlib import Path
    
    start_time = time.time()

    config = {
        'exclude_sections': ['References'],  # Sections to exclude
    }
    
    # Select prompt template based on configuration
    prompt_template = prompt_templates_robotics if use_robotics_prompts else prompt_templates
    
    try:
        # Validate input parameters
        if not arxiv_id and not pdf_path:
            raise ValueError("Either arxiv_id or pdf_path must be provided")
        
        if arxiv_id and pdf_path:
            if verbose:
                print("️  Both arxiv_id and pdf_path provided, using arxiv_id")
        
        # === STEP 0: Check for existing markdown files ===
        existing_md_path = None
        if not force_reprocess:
            # Determine expected markdown file path
            paper_identifier = arxiv_id or Path(pdf_path).stem
            if not ocr_output_dir:
                ocr_output_dir = os.path.join("results", paper_identifier.replace('/', '_'))
            
            # Check for existing markdown file
            expected_md_name = f"{paper_identifier.replace('/', '_')}.md"
            expected_md_path = os.path.join(ocr_output_dir, expected_md_name)
            
            if os.path.exists(expected_md_path):
                # Check file size to ensure it's not empty
                file_size = os.path.getsize(expected_md_path)
                if file_size > 1000:  # At least 1KB
                    existing_md_path = expected_md_path
                    if verbose:
                        print(f"\n Found existing markdown file: {expected_md_path}")
                        print(f"   Size: {file_size / 1024:.1f} KB")
                        print(f"   Skipping download and OCR processing...")
                    
                    pipeline_results['skipped_download'] = True
                    pipeline_results['skipped_ocr'] = True
                    pipeline_results['existing_md_used'] = existing_md_path
                    pipeline_results['md_path'] = existing_md_path
                elif verbose:
                    print(f"\n️ Found existing markdown file but it's too small ({file_size} bytes), will reprocess")
            elif verbose:
                print(f"\n No existing markdown file found at: {expected_md_path}")
                print(f"   Will proceed with download and OCR processing...")
        elif verbose:
            print(f"\n Force reprocess enabled - will download and process regardless of existing files")
        
        # === STEP 1: Paper Acquisition ===
        if verbose:
            print("\n" + "="*60)
            print(" STARTING COMPLETE PIPELINE")
            print("="*60)
        
        if existing_md_path:
            # Skip download, use existing markdown
            if verbose:
                print(f"\n Step 1: Using existing markdown file")
            
            # Create minimal metadata for existing file
            paper_title = Path(existing_md_path).stem
            metadata = {
                'title': paper_title,
                'authors': ['Unknown'],
                'published': None,
                'categories': [],
                'summary': ''
            }
            # Try to find corresponding PDF if it exists
            if arxiv_id:
                potential_pdf = os.path.join(download_dir, f"{arxiv_id}.pdf")
                if os.path.exists(potential_pdf):
                    pipeline_results['pdf_path'] = potential_pdf
            elif pdf_path and os.path.exists(pdf_path):
                pipeline_results['pdf_path'] = pdf_path
        elif arxiv_id:
            temp_config = ARXIV_CONFIG.copy()
            temp_config['arxiv_id'] = arxiv_id
            temp_config['download_dir'] = download_dir
            temp_config['existing_pdf'] = None
            
            downloaded_path, metadata = download_arxiv_paper_for_pipeline(temp_config)
            
            if not downloaded_path:
                raise Exception(f"Failed to download paper from arXiv: {arxiv_id}")
            
            pipeline_results['pdf_path'] = downloaded_path
            paper_title = metadata.get('title', f'arXiv-{arxiv_id}')
            
            if verbose:
                print(f" Downloaded: {paper_title[:80]}...")
                print(f"   Size: {os.path.getsize(downloaded_path) / (1024*1024):.1f} MB")
        else:
            if verbose:
                print(f"\n Step 1: Using existing PDF: {pdf_path}")
            
            if not os.path.exists(pdf_path):
                raise FileNotFoundError(f"PDF file not found: {pdf_path}")
            
            pipeline_results['pdf_path'] = pdf_path
            paper_title = Path(pdf_path).stem
            
            # Create minimal metadata
            metadata = {
                'title': paper_title,
                'authors': ['Unknown'],
                'published': None,
                'categories': [],
                'summary': ''
            }
        
        # === STEP 2: Enhanced OCR Processing ===
        if run_ocr and not existing_md_path:
            if verbose:
                ocr_type = "Enhanced Image Annotations" if use_enhanced_ocr else "Standard OCR"
                print(f"\n Step 2: {ocr_type}")

            
            
            # Set up configuration for OCR
            paper_identifier = arxiv_id or Path(pdf_path).stem
            if not ocr_output_dir:
                ocr_output_dir = os.path.join("results", paper_identifier.replace('/', '_'))
            
            if use_enhanced_ocr:
                # Use enhanced OCR with optional detailed image annotations
                md_path, json_path, annotations = process_pdf_with_enhanced_ocr(
                    pdf_path=pipeline_results['pdf_path'],
                    output_dir=ocr_output_dir,
                    max_pages=max_annotation_pages,
                    include_images=False,  # Use annotations instead of images
                    use_annotations=use_annotations,  # Pass the flag
                    verbose=verbose
                    
                )
                
                if annotations and use_annotations:
                    pipeline_results['image_annotations'] = annotations
                    pipeline_results['image_annotations_count'] = annotations.get('total_annotations', 0)
                    
                    if verbose:
                        print(f"  - Image annotations: {pipeline_results['image_annotations_count']}")
                        # Show annotation breakdown by type
                        annotation_types = annotations.get('annotation_types', {})
                        for img_type, count in sorted(annotation_types.items()):
                            print(f"    - {img_type}: {count}")
                elif verbose and use_annotations:
                    print(f"  - Image annotations: 0 (fallback to standard OCR used)")
                elif verbose:
                    print(f"  - Image annotations: disabled in config")
            else:
                # Use standard OCR processing
                temp_config = {
                    'pdf_path': pipeline_results['pdf_path'],
                    'output_dir': ocr_output_dir,
                    'process_pdf': True,
                    'save_images': True
                }
                md_path, json_path = run_ocr_processing(temp_config)
            
            if not md_path:
                raise Exception("OCR processing failed")
            
            pipeline_results['md_path'] = md_path
            pipeline_results['ocr_results'] = json_path
            
            if verbose:
                print(f" OCR completed")
                print(f"   Markdown saved: {md_path}")
                if use_enhanced_ocr and use_annotations:
                    print(f"   Image annotations: {pipeline_results['image_annotations_count']}")
                elif use_enhanced_ocr:
                    print(f"   Image annotations: disabled")
        elif existing_md_path:
            if verbose:
                print(f"\n Step 2: Skipping OCR (using existing markdown)")
        elif not run_ocr:
            if verbose:
                print(f"\n Step 2: OCR processing disabled")
        
        # === STEP 3: Database Storage (Paper) ===
        if verbose:
            print(f"\n Step 3: Storing paper in database")
        
        paper_id = store_paper_from_md_file(
            pipeline_results['md_path'],
            arxiv_id=arxiv_id or Path(pdf_path).stem,
            title=metadata['title']
        )
        
        if not paper_id:
            raise Exception("Failed to store paper in database")
        
        pipeline_results['paper_id'] = paper_id
        
        if verbose:
            print(f" Paper stored with ID: {paper_id}")
        
        # === STEP 4: LLM Analysis ===
        if run_analysis and pipeline_results['md_path']:
            if verbose:
                print(f"\n Step 4: LLM Analysis")
            
            total_runs = deepseek_runs + gemini_runs 
            if verbose:
                print(f"   Planning {total_runs} total analysis runs")
                print(f"   DeepSeek: {deepseek_runs}, Gemini: {gemini_runs}")
            
            # Use the existing token analysis function
            token_count, needs_chunking = analyze_document_tokens(
                pipeline_results['md_path'], 
                provider='deepseek',
                max_tokens=max_chunk_size
            )
            
            if verbose:
                print(f"   Document tokens: {token_count:,}")
                print(f"   Needs chunking: {'Yes' if needs_chunking else 'No'}")
            
            # DeepSeek Analysis
            if deepseek_runs > 0:
                if verbose:
                    print(f"\n    Running {deepseek_runs} DeepSeek analysis(es)")
                
                for run_idx in range(deepseek_runs):
                    try:
                        run_temp = temperature + (run_idx * 0.05)  # Slight variation
                        
                        # Use the existing analyze_and_store function directly
                        if needs_chunking:
                            chunked_analyser = create_chunked_analyser(
                                base_analyser=DeepSeekAnalyser(prompt_template=prompt_template),
                                provider='deepseek',
                                max_tokens=max_chunk_size,
                                progressive=True,
                                template_module=prompt_template
                            )
                            if verbose:
                                print(f"      Using chunked analyzer for DeepSeek (Run {run_idx + 1})")

                                        # Set excluded sections
                            if hasattr(chunked_analyser, 'exclude_sections') and config['exclude_sections']:
                                chunked_analyser.exclude_sections = config['exclude_sections']
                                print(f"  - Excluding sections: {', '.join(config['exclude_sections'])}")

                            analyzer = chunked_analyser
                        else:
                            analyzer = DeepSeekAnalyser(prompt_template=prompt_template)
                        
                        if verbose:
                            print(f"      Running DeepSeek analysis (Run {run_idx + 1}) with temperature {run_temp:.2f}")

                        paper_id_result, model_id, run_ids = analyze_and_store(
                            analyzer,
                            pipeline_results['md_path'],
                            "deepseek-chat",
                            "DeepSeek",
                            arxiv_id=arxiv_id,
                            temperature=run_temp
                        )
                        
                        if run_ids:
                            pipeline_results['analysis_results']['deepseek'].extend(run_ids)
                            pipeline_results['total_extractions'] += len(run_ids)
                            
                            if verbose:
                                print(f"      Run {run_idx + 1}: {len(run_ids)} extractions")
                        else:
                            if verbose:
                                print(f"      Run {run_idx + 1}: Failed")
                            pipeline_results['errors'].append(f"DeepSeek run {run_idx + 1} failed")
                    
                    except Exception as e:
                        error_msg = f"DeepSeek run {run_idx + 1} error: {str(e)}"
                        pipeline_results['errors'].append(error_msg)
                        if verbose:
                            print(f"      Run {run_idx + 1}: {str(e)}")
            
            # Gemini Analysis
            if gemini_runs > 0:
                if verbose:
                    print(f"\n    Running {gemini_runs} Gemini analysis(es)")
                
                for run_idx in range(gemini_runs):
                    try:
                        run_temp = temperature + (run_idx * 0.05)
                        
                        # Use the existing analyze_and_store function directly
                        analyzer = GeminiAnalyser(prompt_template=prompt_template)
                        paper_id_result, model_id, run_ids = analyze_and_store(
                            analyzer,
                            pipeline_results['md_path'],
                            "gemini-pro",
                            "Google",
                            temperature=run_temp
                        )
                        
                        if run_ids:
                            pipeline_results['analysis_results']['gemini'].extend(run_ids)
                            pipeline_results['total_extractions'] += len(run_ids)
                            
                            if verbose:
                                print(f"      Run {run_idx + 1}: {len(run_ids)} extractions")
                        else:
                            if verbose:
                                print(f"      Run {run_idx + 1}: Failed")
                            pipeline_results['errors'].append(f"Gemini run {run_idx + 1} failed")
                    
                    except Exception as e:
                        error_msg = f"Gemini run {run_idx + 1} error: {str(e)}"
                        pipeline_results['errors'].append(error_msg)
                        if verbose:
                            print(f"      Run {run_idx + 1}: {str(e)}")
        
        # === PIPELINE COMPLETION ===
        pipeline_results['success'] = True
        pipeline_results['processing_time'] = time.time() - start_time
        
        if verbose:
            print(f"\n" + "="*60)
            print(f" ENHANCED PIPELINE COMPLETED SUCCESSFULLY")
            print(f"="*60)
            print(f" Summary:")
            print(f"   Paper ID: {pipeline_results['paper_id']}")
            print(f"   Processing time: {pipeline_results['processing_time']:.1f}s")
            if pipeline_results['skipped_download']:
                print(f"   ⏭️  Skipped download (existing file used)")
            if pipeline_results['skipped_ocr']:
                print(f"   ⏭️  Skipped OCR (existing markdown used)")
            if use_enhanced_ocr and use_annotations:
                print(f"   Image annotations: {pipeline_results['image_annotations_count']}")
            print(f"   Total extractions: {pipeline_results['total_extractions']}")
        
        return pipeline_results
        
    except Exception as e:
        pipeline_results['success'] = False
        pipeline_results['processing_time'] = time.time() - start_time
        error_msg = f"Pipeline failed: {str(e)}"
        pipeline_results['errors'].append(error_msg)
        
        if verbose:
            print(f"\n PIPELINE FAILED after {pipeline_results['processing_time']:.1f}s")
            print(f"Error: {str(e)}")
        
        return pipeline_results

print(" Enhanced integrated pipeline function updated with existing file check!")

## Example Usage of the Integrated Pipeline

Here are several examples showing how to use the integrated pipeline function with different configurations:

In [ ]:
# Example 1: Complete pipeline with arXiv download and multiple analyses
print("Example 1: Complete pipeline with arXiv paper")
print("="*50)


# Run complete pipeline with arXiv paper
'''
results_example1 = run_complete_pipeline(
    arxiv_id='2408.12570',
    run_ocr=True,
    use_enhanced_ocr=True,
    use_annotations=False,  # SET TO FALSE SINCE ENDPOINT DOESN'T SUPPORT ANNOTATIONS YET
    force_reprocess=False,  # NEW: Check for existing files first
    run_analysis=True,
    deepseek_runs=1,
    gemini_runs=0,
    temperature=0.1,
    verbose=True
)
'''
os.system("clear")  # Clear console for better readability


print(f"\n Example 1 Results:")
print(f"Success: {results_example1['success']}")
print(f"Paper ID: {results_example1['paper_id']}")
print(f"Total processing time: {results_example1['processing_time']:.1f}s")
print(f"Total extractions: {results_example1['total_extractions']}")
print(f"Skipped download: {results_example1['skipped_download']}")
print(f"Skipped OCR: {results_example1['skipped_ocr']}")
if results_example1['existing_md_used']:
    print(f"Used existing MD: {results_example1['existing_md_used']}")


In [ ]:

print("\nExample 4: Batch processing multiple arXiv papers")
print("="*50)


# Import arXiv IDs from a file

#uncomment the line below to specify the file containing arXiv IDs <<<<-------------------------
#arxiv_ids_file = 'arxivIDs.txt'

# Uncomment the line below to import arXiv IDs from a file
# arxiv_papers = import_arxiv_ids(arxiv_ids_file) 


# List of arXiv IDs to process
# arxiv_papers = [
#     '2402.07827',
#     # '2101.12345',  # Add more arXiv IDs as needed
#     # '2012.67890',
# ]

batch_results = []

for i, arxiv_id in enumerate(arxiv_papers, 1):
    print(f"\n Processing paper {i}/{len(arxiv_papers)}: {arxiv_id}")
    print("-" * 40)
    
    try:
        # Run pipeline for each paper
        result = run_complete_pipeline(
            arxiv_id=arxiv_id,
            run_ocr=True,
            run_analysis=True,
            use_enhanced_ocr=False,  # Enhanced OCR with image annotations
            use_annotations=False,  # SET TO FALSE SINCE ENDPOINT DOESN'T SUPPORT ANNOTATIONS YET
            force_reprocess=False,  # NEW: Check for existing files to speed up batch processing
            deepseek_runs=1,
            gemini_runs=1,
            temperature=0.1,
            verbose=False  # Reduced verbosity for batch processing
        )
        
        batch_results.append({
            'arxiv_id': arxiv_id,
            'success': result['success'],
            'paper_id': result['paper_id'],
            'extractions': result['total_extractions'],
            'processing_time': result['processing_time'],
            'errors': len(result['errors']),
            'skipped_download': result['skipped_download'],
            'skipped_ocr': result['skipped_ocr']
        })
        
        status = " SUCCESS" if result['success'] else " FAILED"
        skip_info = ""
        if result['skipped_download'] and result['skipped_ocr']:
            skip_info = " (used existing)"
        elif result['skipped_download']:
            skip_info = " (skipped download)"
        elif result['skipped_ocr']:
            skip_info = " (skipped OCR)"
        print(f"{status}{skip_info} - {result['total_extractions']} extractions in {result['processing_time']:.1f}s")
        
    except Exception as e:
        print(f" FAILED - {str(e)}")
        batch_results.append({
            'arxiv_id': arxiv_id,
            'success': False,
            'error': str(e)
        })

# Summary of batch processing
print(f"\n Batch Processing Summary:")
print(f"Total papers processed: {len(batch_results)}")
successful = sum(1 for r in batch_results if r.get('success', False))
skipped_downloads = sum(1 for r in batch_results if r.get('skipped_download', False))
skipped_ocr = sum(1 for r in batch_results if r.get('skipped_ocr', False))
print(f"Successful: {successful}")
print(f"Failed: {len(batch_results) - successful}")
print(f"Skipped downloads: {skipped_downloads}")
print(f"Skipped OCR: {skipped_ocr}")

if successful > 0:
    total_extractions = sum(r.get('extractions', 0) for r in batch_results if r.get('success'))
    total_time = sum(r.get('processing_time', 0) for r in batch_results if r.get('success'))
    print(f"Total extractions: {total_extractions}")
    print(f"Total processing time: {total_time:.1f}s")
    print(f"Average time per paper: {total_time/successful:.1f}s")
    print(f"\n Time saved by skipping existing files: {skipped_downloads + skipped_ocr} operations")


## SETUP THE PIPELINE

In [ ]:
# This code imports arxiv ids from the arxivIDs.txt file and and saves them in a list.
def import_arxiv_ids(file_path):
    """Import arXiv IDs from a text file."""
    arxiv_ids = []
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#'):  # Ignore empty lines and comments
                    arxiv_ids.append(line)
        print(f" Imported {len(arxiv_ids)} arXiv IDs from {file_path}")

        #remove duplicates
        arxiv_ids = list(set(arxiv_ids))
        print(f" Removed duplicates, {len(arxiv_ids)} unique arXiv IDs available.")
    except Exception as e:
        print(f" Error importing arXiv IDs: {e}")
    return arxiv_ids

In [ ]:
# Example usage of the complete pipeline function with improved parallel processing
print("\nExample 5: Improved Parallel Processing - Sequential Download + Parallel Processing")
print("="*80)  
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import clear_output
import time

def download_papers_sequentially(arxiv_ids, download_dir="downloaded_papers"):
    """Download all papers sequentially to avoid arXiv rate limiting."""
    print(f"\n Phase 1: Sequential Download of {len(arxiv_ids)} papers")
    print("-" * 60)
    
    download_results = {}
    successful_downloads = 0
    
    for i, arxiv_id in enumerate(arxiv_ids, 1):
        print(f"\n Downloading {i}/{len(arxiv_ids)}: {arxiv_id}", end=" ... ")
        
        try:
            # Check if already downloaded
            expected_pdf = os.path.join(download_dir, f"{arxiv_id}.pdf")
            if os.path.exists(expected_pdf):
                file_size = os.path.getsize(expected_pdf) / (1024 * 1024)
                print(f" Already exists ({file_size:.1f} MB)")
                download_results[arxiv_id] = {
                    'success': True,
                    'pdf_path': expected_pdf,
                    'skipped': True
                }
                successful_downloads += 1
                if i % 20 == 0:
                    clear_output(wait=True)
                continue
            
            # Download the paper
            temp_config = {
                'download_from_arxiv': True,
                'arxiv_id': arxiv_id,
                'download_dir': download_dir,
                'existing_pdf': None
            }
            
            start_time = time.time()
            pdf_path, metadata = download_arxiv_paper_for_pipeline(temp_config)
            elapsed_time = time.time() - start_time
            
            if pdf_path:
                file_size = os.path.getsize(pdf_path) / (1024 * 1024)
                print(f" Downloaded ({file_size:.1f} MB, {elapsed_time:.1f}s)")
                download_results[arxiv_id] = {
                    'success': True,
                    'pdf_path': pdf_path,
                    'metadata': metadata,
                    'skipped': False
                }
                successful_downloads += 1
                
                # Small delay to be respectful to arXiv
                time.sleep(0.5)
                
            else:
                print(f" Failed ({elapsed_time:.1f}s)")
                download_results[arxiv_id] = {
                    'success': False,
                    'error': 'Download failed',
                    'skipped': False
                }

            if i % 5 == 0:  # Clear output every 5 downloads
                clear_output(wait=True)
                
        except Exception as e:
            print(f" Error: {str(e)[:50]}...")
            download_results[arxiv_id] = {
                'success': False,
                'error': str(e),
                'skipped': False
            }
    
    print(f"\n Download Summary:")
    print(f"  Total papers: {len(arxiv_ids)}")
    print(f"  Successfully downloaded: {successful_downloads}")
    print(f"  Failed: {len(arxiv_ids) - successful_downloads}")
    skipped = sum(1 for r in download_results.values() if r.get('skipped', False))
    print(f"  Already existed: {skipped}")
    print(f"  Newly downloaded: {successful_downloads - skipped}")
    
    return download_results

def process_paper_parallel(arxiv_id, pdf_info):
    """Process a single paper with OCR and analysis (for parallel execution)."""
    try:
        if not pdf_info['success']:
            return {
                'arxiv_id': arxiv_id,
                'success': False,
                'error': f"PDF not available: {pdf_info.get('error', 'Unknown')}"
            }
        
        # Run pipeline with existing PDF (skip download phase)
        result = run_complete_pipeline(
            arxiv_id=arxiv_id,
            pdf_path=pdf_info['pdf_path'],  # Use pre-downloaded PDF
            run_ocr=True,
            run_analysis=True,
            use_enhanced_ocr=True,
            use_annotations=False,
            force_reprocess=False,
            deepseek_runs=1,
            gemini_runs=0,
            temperature=0.1,
            verbose=False,  # Reduced verbosity for parallel processing
            max_chunk_size=65536,  # <<<-------  change this to adjust chunk size in parallel processing ---------
        )
        
        return {
            'arxiv_id': arxiv_id,
            'success': result['success'],
            'paper_id': result['paper_id'],
            'extractions': result['total_extractions'],
            'processing_time': result['processing_time'],
            'errors': len(result['errors']),
            'skipped_download': result['skipped_download'],
            'skipped_ocr': result['skipped_ocr'],
            'pdf_existed': pdf_info.get('skipped', False)
        }
        
    except Exception as e:
        return {
            'arxiv_id': arxiv_id,
            'success': False,
            'error': str(e)
        }

## LOAD THE PAPERS

In [ ]:
# Main improved parallel processing
arxiv_ids_file = 'arxivIDs.txt'
arxiv_papers = import_arxiv_ids(arxiv_ids_file)
print(f"\n Starting improved parallel processing for {len(arxiv_papers)} arXiv papers...")

# Phase 1: Sequential Download
download_results = download_papers_sequentially(arxiv_papers)
# Filter successful downloads for processing
papers_to_process = {arxiv_id: info for arxiv_id, info in download_results.items() if info['success']}


## RUN THE PIPELINE

In [ ]:

if not papers_to_process:
    print("\n No papers available for processing. All downloads failed.")
else:
    print(f"\n Phase 2: Parallel Processing of {len(papers_to_process)} papers")
    print("-" * 60)
    
    # Phase 2: Parallel Processing (OCR + Analysis)
    parallel_results = []
    
    # Use fewer workers for processing to avoid overwhelming the system
    max_workers = min(16, len(papers_to_process))  # Reduced from 8 to 4
    print(f"Using {max_workers} parallel workers for processing...")
    
    #clear output every 5 papers to keep console clean
    clear_every = 5
    count = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_arxiv = {
            executor.submit(process_paper_parallel, arxiv_id, pdf_info): arxiv_id 
            for arxiv_id, pdf_info in papers_to_process.items()
        }

        # clear output every 10 papers
        for future in future_to_arxiv:
            count += 1
            if count % clear_every == 0:
                clear_output(wait=True)
                print(f"Processing papers... {count}/{len(papers_to_process)} completed")
        
        for future in as_completed(future_to_arxiv):
            arxiv_id = future_to_arxiv[future]
            try:
                result = future.result()
                parallel_results.append(result)
                
                status = " SUCCESS" if result['success'] else " FAILED"
                skip_info = ""
                if result.get('pdf_existed'):
                    skip_info += " (PDF existed)"
                if result.get('skipped_ocr'):
                    skip_info += " (OCR skipped)"
                if result.get('skipped_download'):
                    skip_info += " (Download skipped)"
                
                extractions = result.get('extractions', 0)
                proc_time = result.get('processing_time', 0)
                print(f"{status}{skip_info} - {result['arxiv_id']} - {extractions} extractions in {proc_time:.1f}s")
                
            except Exception as e:
                print(f" PROCESSING FAILED for {arxiv_id} - {str(e)}")
                parallel_results.append({
                    'arxiv_id': arxiv_id,
                    'success': False,
                    'error': str(e)
                })

    # Final Summary
    print(f"\n Final Processing Summary:")
    print(f"="*60)
    print(f"Total papers requested: {len(arxiv_papers)}")
    print(f"Successfully downloaded: {len(papers_to_process)}")
    print(f"Successfully processed: {sum(1 for r in parallel_results if r.get('success', False))}")
    print(f"Failed processing: {sum(1 for r in parallel_results if not r.get('success', False))}")
    
    if parallel_results:
        successful_results = [r for r in parallel_results if r.get('success', False)]
        if successful_results:
            total_extractions = sum(r.get('extractions', 0) for r in successful_results)
            total_time = sum(r.get('processing_time', 0) for r in successful_results)
            avg_time = total_time / len(successful_results) if successful_results else 0
            
            print(f"\n Performance Metrics:")
            print(f"  Total extractions: {total_extractions}")
            print(f"  Total processing time: {total_time:.1f}s")
            print(f"  Average time per paper: {avg_time:.1f}s")
            
            # Show efficiency gains
            pdfs_existed = sum(1 for r in parallel_results if r.get('pdf_existed', False))
            ocr_skipped = sum(1 for r in parallel_results if r.get('skipped_ocr', False))
            print(f"\n Efficiency Gains:")
            print(f"  PDFs that already existed: {pdfs_existed}")
            print(f"  OCR processing skipped: {ocr_skipped}")
            print(f"  Download failures avoided by sequential approach: Prevented rate limiting")

        else:
            print(f"\n No successful processing results found.")

print(f"\n Improved parallel processing completed!")
print(f" Benefits of this approach:")
print(f"  - Sequential downloads prevent arXiv rate limiting")
print(f"  - Parallel processing speeds up OCR and analysis")
print(f"  - Better error handling and progress tracking")
print(f"  - Efficient reuse of existing files")


In [ ]:
#function which counts the number of tokens using the tokenizer per paper in a given dictionary of papers coming from download_papers_sequentially()
def count_tokens_in_papers(papers_dict):
    print(papers_dict)
    tokenizer = get_tokenizer("deepseek", max_tokens=99999999999999)  # Use a very high max_tokens to avoid truncation
    print(" Tokenizer obtained successfully, counting tokens in papers...")
    if tokenizer:
        token_counts = {}
        for arxiv_id, info in papers_dict.items():
            print(f" Counting tokens for {arxiv_id}...")

            pdf_path = info.get('pdf_path')
            

            # Determine expected markdown file path
            paper_identifier = arxiv_id or Path(pdf_path).stem
           
            ocr_output_dir = os.path.join("results", paper_identifier.replace('/', '_'))
            
            # Check for existing markdown file
            expected_md_name = f"{paper_identifier.replace('/', '_')}.md"
            expected_md_path = os.path.join(ocr_output_dir, expected_md_name)
            
            if os.path.exists(expected_md_path):
                # Check file size to ensure it's not empty
                file_size = os.path.getsize(expected_md_path)
                if file_size > 1000:  # At least 1KB
                    existing_md_path = expected_md_path

                    print(f"  - Found existing markdown file: {expected_md_path}")
                    print(f"   Size: {file_size / 1024:.1f} KB")
                    print(f"   Counting tokens in markdown file...")
                    tmp_file = open(expected_md_path, 'r', encoding='utf-8')
                    prompt_length = tokenizer.count_tokens(tmp_file.read())
                    tmp_file.close()
                    print(f"  - Found {prompt_length} tokens in markdown file")
                    token_counts[arxiv_id] = prompt_length
                else:
                    print(f"  - Found existing markdown file but it's too small ({file_size} bytes), will reprocess")
                    existing_md_path = None
            else:
                print(f"  - No existing markdown file found at: {expected_md_path}")
           
                
        return token_counts

    else:
        print(" Failed to get tokenizer")
        return None
    
# count tokens from download_results
print("\n Counting tokens in downloaded papers...")
token_counts = count_tokens_in_papers(papers_to_process)


In [ ]:
if token_counts:
    for arxiv_id, count in token_counts.items():
        pass
       #print(f" {arxiv_id}: {count} tokens")
        
    #average tokens per paper
    avg_tokens = sum(token_counts.values()) / len(token_counts)
    print(f"Average tokens per paper: {avg_tokens:.2f}")
    #num of papers with less than 4k,8k,16k, 32k, 64k, 128k tokens
    num_less_than_4k = sum(1 for c in token_counts.values() if c < 4000)
    num_less_than_8k = sum(1 for c in token_counts.values() if c < 8000)
    num_less_than_16k = sum(1 for c in token_counts.values() if c < 16000)
    num_less_than_32k = sum(1 for c in token_counts.values() if c < 32000)
    num_less_than_64k = sum(1 for c in token_counts.values() if c < 64000)
    num_less_than_128k = sum(1 for c in token_counts.values() if c < 128000)
    print(f"Total number of papers: {len(token_counts)}")
    print(f"Number of papers with less than 4k tokens: {num_less_than_4k}")
    print(f"Number of papers with less than 8k tokens: {num_less_than_8k}")
    print(f"Number of papers with less than 16k tokens: {num_less_than_16k}")
    print(f"Number of papers with less than 32k tokens: {num_less_than_32k}")
    print(f"Number of papers with less than 64k tokens: {num_less_than_64k}")
    print(f"Number of papers with less than 128k tokens: {num_less_than_128k}")
    print(token_counts)

else:
    print(" Failed to count tokens.")

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Your example dictionary


# Step 1: Group lengths by 5k buckets
def group_by_5k(length):
    return (length // 5000) * 5000

grouped = [group_by_5k(length) for length in token_counts.values()]
bucket_counts = Counter(grouped)

# Step 2: Sort buckets for plotting
sorted_buckets = sorted(bucket_counts.items())

# Step 3: Unzip keys and counts
bucket_labels, counts = zip(*sorted_buckets)

# Step 4: Format labels for readability
labels = [f"{k}-{k+4999}" for k in bucket_labels]

# Step 5: Plot histogram
plt.figure(figsize=(10, 6))
plt.bar(labels, counts, width=0.8, align='center')
plt.xticks(rotation=45)
plt.xlabel("Token Length Range")
plt.ylabel("Count")
plt.title("Histogram of Token Lengths (Grouped by 5k)")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

token_lengths_values = list(token_counts.values())

# Plot histogram of log-transformed lengths
log_lengths = np.log10(token_lengths_values)

plt.hist(log_lengths, bins=20, edgecolor='black')
plt.xlabel("Log10(Token Length)")
plt.ylabel("Count")
plt.title("Log-Transformed Token Length Distribution")
plt.show()